# 1. Import Data

In [119]:
# Global Import and Directories
import sys
sys.path.append("..")
import Setup
from Setup import *

## 1.1. Data Thuringia

In [14]:
# Load main data set
df_thuringia = pd.read_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "02_Analysis" / "Thuringia.csv",
    dtype={9: str}
)
df_thuringia["date"] = pd.to_datetime(df_thuringia["date"])

/var/tmp/pbs.17081774.hpc-batch/ipykernel_44569/3416999999.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_thuringia = pd.read_csv(


In [15]:
# Load travel time data
tt_thuringia = pd.read_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "01_OSMR" / "stations_travel_times_thuringia.csv.gz"
)

# Test if there is a travel time for all combinations
tt_thuringia.isna().sum()

src_uuid           0
dst_uuid           0
travel_time_min    0
dtype: int64

In [16]:
tt_thuringia = tt_thuringia[tt_thuringia["travel_time_min"] < 10]
tt_thuringia["travel_time_min"].describe()

count    6719.000000
mean        6.279962
std         2.494912
min         0.013333
25%         4.441667
50%         6.611667
75%         8.404167
max         9.998333
Name: travel_time_min, dtype: float64

## 1.2. Data Hesse

In [17]:
# Load main data set
df_hesse = pd.read_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "02_Analysis" / "Hesse.csv",
    dtype={9: str}
)
df_hesse["date"] = pd.to_datetime(df_hesse["date"])

/var/tmp/pbs.17081774.hpc-batch/ipykernel_44569/1468913650.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_hesse = pd.read_csv(


In [18]:
# Load travel time data
tt_hesse = pd.read_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "01_OSMR" / "stations_travel_times_hesse.csv.gz"
)

# Test if there is a travel time for all combinations
tt_hesse.isna().sum()

src_uuid           0
dst_uuid           0
travel_time_min    0
dtype: int64

In [19]:
tt_hesse = tt_hesse[tt_hesse["travel_time_min"] < 10]
tt_hesse["travel_time_min"].describe()

count    23044.000000
mean         6.101753
std          2.900616
min          0.000000
25%          4.281667
50%          6.776667
75%          8.470417
max          9.998333
Name: travel_time_min, dtype: float64

# 2. Prepare Data

In [83]:
# =============================
# a) Parameters
# =============================

STATION_COL = "station_uuid"
LAT_COL = "latitude"
LON_COL = "longitude"
TIME_COL = "date"

# Default-Targets
TARGET_COLS_DEFAULT = ["e5"]
FEATURE_COLS_DEFAULT = ["e5"]  # standard: Targets = Features
BRAND_COL = "brand"
HIGHWAY_COL = "Highway_Identifier"  
REGION_COL_DEFAULT = "in_thuringia"
TT_SRC_COL = "src_uuid"
TT_DST_COL = "dst_uuid"
TT_TIME_COL = "travel_time_min"

NEIGHBOUR = 5
Travel_Time = 10

SEED = 42
EMBED_DIM = 32
HIDDEN_DIM = 64
LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 100
ATTN_HEADS_DEFAULT = 4
PATIENCE = 10
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

WINDOW_SIZE = 672 # high value due to different encoder
HORIZON_STEPS = 4  # Horizont per Target
LENGTH_ENCODER_DEFAULT = 16 # Target Length after Strides


# =============================
# b) Helper functions
# =============================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# =============================
# c) Window-based ST data (multi-step, multi-target)
# =============================

@dataclass
class STWindowDataMulti:
    x: torch.Tensor             # [T, N, F] full time series, F = num_features
    x_mask: torch.Tensor   
    y: torch.Tensor             # [T, N, H * num_targets] H Horizonte für alle Targets
    edge_index: torch.Tensor
    edge_weight: torch.Tensor 
    train_end_idx: np.ndarray   # time indices where windows end (train)
    val_end_idx: np.ndarray     # time indices (validation)
    test_end_idx: np.ndarray    # time indices (test)
    valid_nodes: np.ndarray     # node indices (z.B. Thüringen)
    times: np.ndarray           # sorted timestamps
    meta: pd.DataFrame          # station metadata (uuid, lat, lon, node_id)
    feature_cols: List[str]     # relevant Feature-Spalten
    target_cols: List[str]      # relevant Target-Spalten
    time_feat: np.ndarray       # [T, time_in_dim], z.B. weekday, holiday, time_sin
    condat: Optional[torch.Tensor]  # [N, cond_in_dim]


def make_window_data_multi_from_df(
    df: pd.DataFrame,
    window_size: int = WINDOW_SIZE,
    horizon_steps: int = HORIZON_STEPS,
    feature_cols: Optional[List[str]] = None,
    target_cols: Optional[List[str]] = None,
    stride: int = 1, 
    neighbour_k: int = NEIGHBOUR,
    travel_time: int = Travel_Time,
    region_col: str = REGION_COL_DEFAULT,      
    tt: pd.DataFrame | None = None
) -> STWindowDataMulti:

    if target_cols is None:
        target_cols = TARGET_COLS_DEFAULT
    if feature_cols is None:
        feature_cols = FEATURE_COLS_DEFAULT

    # Necessary Columns
    base_cols = [STATION_COL, LAT_COL, LON_COL, TIME_COL, region_col, BRAND_COL]
    all_val_cols = sorted(set(feature_cols) | set(target_cols))
    needed = base_cols + all_val_cols
    assert all(c in df.columns for c in needed), f"Missing columns: {set(needed) - set(df.columns)}"

    df = df.copy()

    # Stations + node ids
    stations = (
    df[[STATION_COL, LAT_COL, LON_COL, BRAND_COL, HIGHWAY_COL]]
    .drop_duplicates(subset=[STATION_COL])
    .reset_index(drop=True)
    )
    
    stations["node_id"] = np.arange(len(stations))
    station2id = stations.set_index(STATION_COL)["node_id"].to_dict()

    # Time axis
    times = np.sort(df[TIME_COL].unique())
    time2id = {t: i for i, t in enumerate(times)}
    
    # ---- Build time features: weekday, holiday, time_sin ----
    
    weekday_cols = [f"wd_{i}" for i in range(7)]
    time_cols = [f"t_{i}" for i in range(96)]
    
    time_df = (
        df[[TIME_COL, *weekday_cols, "holiday", *time_cols, "Brent_Price"]]
        .drop_duplicates(subset=[TIME_COL])
        .sort_values(TIME_COL)
    )

    assert np.array_equal(time_df[TIME_COL].to_numpy(), times), \
        "The timelines of time_df and times do not match."

    time_feat = time_df[[*weekday_cols, "holiday", *time_cols, "Brent_Price"]].to_numpy(dtype=float)

    N = len(stations)
    T = len(times)
    M = len(target_cols)             # Number Targets
    F = len(feature_cols)            # Number Features
    H = horizon_steps
     
    
    # ---- condat aus brand + highway bauen (One-Hot) ----
    brands = stations[BRAND_COL].astype("category")
    brand_codes = brands.cat.codes.to_numpy()
    n_brands = len(brands.cat.categories)
    brand_oh = np.eye(n_brands, dtype=np.float32)[brand_codes]  # [N, n_brands]
 
    highway_oh = stations[HIGHWAY_COL].astype(bool).astype(np.float32).to_numpy().reshape(-1, 1)

    condat_np = np.concatenate([brand_oh, highway_oh], axis=1)  # [N, n_brands + n_highways]
    condat = torch.tensor(condat_np, dtype=torch.float32)
    

    # -------------------------
    # Spatial graph from tt
    # -------------------------
    if tt is None:
        raise ValueError("tt must be provided to build the travel-time graph.")

    # 1) keep only edges where both endpoints exist + travel time < threshold
    tt_use = tt[
        tt[TT_SRC_COL].isin(station2id)
        & tt[TT_DST_COL].isin(station2id)
        & tt[TT_TIME_COL].notna()
        & (tt[TT_TIME_COL] < travel_time) 
    ][[TT_SRC_COL, TT_DST_COL, TT_TIME_COL]].copy()

    # optional: remove self-loops
    tt_use = tt_use[tt_use[TT_SRC_COL] != tt_use[TT_DST_COL]]

    # 2) keep only K fastest neighbours per src
    K = neighbour_k
    tt_use = (
        tt_use
        .sort_values(TT_TIME_COL)
        .groupby(TT_SRC_COL, as_index=False)
        .head(K)
    )

    # 3) map uuids -> node ids
    src = tt_use[TT_SRC_COL].map(station2id).to_numpy(dtype=np.int64)
    dst = tt_use[TT_DST_COL].map(station2id).to_numpy(dtype=np.int64)
    edge_time = tt_use[TT_TIME_COL].to_numpy(dtype=np.float64)

    edge_index = torch.tensor(
        np.vstack([src, dst]),
        dtype=torch.long
    )

    # travel time -> weight
    sigma = np.median(edge_time)
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0

    edge_weight = torch.tensor(
        np.exp(-(edge_time**2) / (2 * sigma**2)),
        dtype=torch.float32
    ) 
    

    # Tensors
    x = torch.zeros((T, N, F), dtype=torch.float32)                    # [T, N, F]
    x_mask = torch.zeros((T, N, F), dtype=torch.float32) 
    y = torch.full((T, N, H * M), float("nan"), dtype=torch.float32)   # [T, N, H*M]

    # Aggregation per (station, time) across all relevant columns
    df_idx = (
        df[[STATION_COL, TIME_COL] + all_val_cols]
        .groupby([STATION_COL, TIME_COL], as_index=False)
        .mean()
        .sort_values(TIME_COL)
    )

    df_idx["node_id"] = df_idx[STATION_COL].map(station2id)
    df_idx["time_id"] = df_idx[TIME_COL].map(time2id)

    # x[t, n, f] = Features at time t
    for _, row in df_idx.iterrows():
        t = int(row["time_id"])
        n = int(row["node_id"])
        for f_idx, col in enumerate(feature_cols):
            v = row[col]
            if pd.notna(v):
                x[t, n, f_idx] = float(v)
                x_mask[t, n, f_idx] = 1.0

    # y[t, n, m*H + h] = Target m at time t+h+1
    for station, g in df_idx.groupby(STATION_COL):
        g = g.sort_values("time_id")
        node_id = int(g["node_id"].iloc[0])
        t_ids = g["time_id"].to_numpy()

        L = len(t_ids)
        for m, col in enumerate(target_cols):
            vals = g[col].to_numpy()

            for idx in range(L):
                t = int(t_ids[idx])
                if idx + H >= L:
                    break

                future_vals = vals[idx + 1: idx + 1 + H]

                if np.any(np.isnan(future_vals)):
                    continue

                for h in range(H):
                    y[t, node_id, m * H + h] = float(future_vals[h])

    # Time indices where full horizon is available
    effective_T = T - H
    if effective_T < window_size:
        raise ValueError(f"Too few timesteps ({effective_T}) for window size {window_size} and horizon {H}.")

    # Time-based split over 0..effective_T-1
    time_indices = np.arange(effective_T)
    num_total = len(time_indices)
    num_train = int((1.0 - VAL_SPLIT - TEST_SPLIT) * num_total)
    num_val   = int(VAL_SPLIT * num_total)
    num_test  = num_total - num_train - num_val

    train_t = time_indices[:num_train]
    val_t   = time_indices[num_train:num_train + num_val]
    test_t  = time_indices[num_train + num_val:]

    # Window end indices
    min_end = window_size - 1
    train_end_idx = train_t[train_t >= min_end]
    val_end_idx   = val_t[val_t >= min_end]
    test_end_idx  = test_t[test_t >= min_end]
    
    # stride
    train_end_idx = train_end_idx[::stride]
    val_end_idx   = val_end_idx[::stride]
    test_end_idx  = test_end_idx[::stride]

    # Thuringia mask
    reg_mask = (
        df[[STATION_COL, region_col]]
        .drop_duplicates()
        .set_index(STATION_COL)[region_col]
    )

    stations = stations.join(reg_mask, on=STATION_COL)

    # robust gegen True/False und 0/1
    valid_nodes = stations.index[stations[region_col].astype(int) == 1].to_numpy()

    if valid_nodes.size == 0:
        raise ValueError(f"No nodes with {region_col} == 1 found.")


    meta = stations[[STATION_COL, LAT_COL, LON_COL, "node_id"]].copy()

    return STWindowDataMulti(
        x=x,
        x_mask=x_mask,
        y=y,
        edge_index=edge_index,
        edge_weight=edge_weight, 
        train_end_idx=train_end_idx,
        val_end_idx=val_end_idx,
        test_end_idx=test_end_idx,
        valid_nodes=valid_nodes,
        times=times,
        meta=meta,
        feature_cols=feature_cols,
        target_cols=target_cols,
        time_feat=time_feat,
        condat=condat,
    )

# 3. Create Data Windows, Thuringia

## 3.1. Data Windows E5

In [102]:
# 4 Neighbours
data_E5_4N_Thuringia = make_window_data_multi_from_df(df_thuringia, tt=tt_thuringia, region_col="in_thuringia", neighbour_k=4, travel_time=10, feature_cols=["e5"], target_cols=["e5"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "01_Thuringia" / "01_E5" / "02_Time" / "data_Time_4N.pt"

torch.save({
    "x": data_E5_4N_Thuringia.x,
    "x_mask": data_E5_4N_Thuringia.x_mask,
    "y": data_E5_4N_Thuringia.y,
    "edge_index": data_E5_4N_Thuringia.edge_index,
    "edge_weight": data_E5_4N_Thuringia.edge_weight,
    "time_feat": data_E5_4N_Thuringia.time_feat,
    "condat": data_E5_4N_Thuringia.condat,
    "train_end_idx": data_E5_4N_Thuringia.train_end_idx,
    "val_end_idx": data_E5_4N_Thuringia.val_end_idx,
    "test_end_idx": data_E5_4N_Thuringia.test_end_idx,
    "valid_nodes": data_E5_4N_Thuringia.valid_nodes,
    "target_cols": data_E5_4N_Thuringia.target_cols,
},  save_path)

In [84]:
# 5 Neighbours
data_E5_5N_Thuringia = make_window_data_multi_from_df(df_thuringia, tt=tt_thuringia, region_col="in_thuringia", neighbour_k=5, travel_time=10, feature_cols=["e5"], target_cols=["e5"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "01_Thuringia" / "01_E5" / "02_Time" / "data_Time_5N.pt"

torch.save({
    "x": data_E5_5N_Thuringia.x,
    "x_mask": data_E5_5N_Thuringia.x_mask,
    "y": data_E5_5N_Thuringia.y,
    "edge_index": data_E5_5N_Thuringia.edge_index,
    "edge_weight": data_E5_5N_Thuringia.edge_weight,
    "time_feat": data_E5_5N_Thuringia.time_feat,
    "condat": data_E5_5N_Thuringia.condat,
    "train_end_idx": data_E5_5N_Thuringia.train_end_idx,
    "val_end_idx": data_E5_5N_Thuringia.val_end_idx,
    "test_end_idx": data_E5_5N_Thuringia.test_end_idx,
    "valid_nodes": data_E5_5N_Thuringia.valid_nodes,
    "target_cols": data_E5_5N_Thuringia.target_cols,
},  save_path)

In [103]:
# 6 Neighbours
data_E5_6N_Thuringia = make_window_data_multi_from_df(df_thuringia, tt=tt_thuringia, region_col="in_thuringia", neighbour_k=6, travel_time=10, feature_cols=["e5"], target_cols=["e5"])
save_path = Setup.DATA_SCRATCH /"02_Processed_Data" / "03_Data_Windows" / "01_Thuringia" / "01_E5" / "02_Time" / "data_Time_6N.pt"

torch.save({
    "x": data_E5_6N_Thuringia.x,
    "x_mask": data_E5_6N_Thuringia.x_mask,
    "y": data_E5_6N_Thuringia.y,
    "edge_index": data_E5_6N_Thuringia.edge_index,
    "edge_weight": data_E5_6N_Thuringia.edge_weight,
    "time_feat": data_E5_6N_Thuringia.time_feat,
    "condat": data_E5_6N_Thuringia.condat,
    "train_end_idx": data_E5_6N_Thuringia.train_end_idx,
    "val_end_idx": data_E5_6N_Thuringia.val_end_idx,
    "test_end_idx": data_E5_6N_Thuringia.test_end_idx,
    "valid_nodes": data_E5_6N_Thuringia.valid_nodes,
    "target_cols": data_E5_6N_Thuringia.target_cols,
},  save_path)

## 3.2. Data Windows E10

In [104]:
# 4 Neighbours
data_E10_4N_Thuringia = make_window_data_multi_from_df(df_thuringia, tt=tt_thuringia, region_col="in_thuringia", neighbour_k=4, travel_time=10, feature_cols=["e10"], target_cols=["e10"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "01_Thuringia" / "02_E10" / "02_Time" / "data_Time_4N.pt"

torch.save({
    "x": data_E10_4N_Thuringia.x,
    "x_mask": data_E10_4N_Thuringia.x_mask,
    "y": data_E10_4N_Thuringia.y,
    "edge_index": data_E10_4N_Thuringia.edge_index,
    "edge_weight": data_E10_4N_Thuringia.edge_weight,
    "time_feat": data_E10_4N_Thuringia.time_feat,
    "condat": data_E10_4N_Thuringia.condat,
    "train_end_idx": data_E10_4N_Thuringia.train_end_idx,
    "val_end_idx": data_E10_4N_Thuringia.val_end_idx,
    "test_end_idx": data_E10_4N_Thuringia.test_end_idx,
    "valid_nodes": data_E10_4N_Thuringia.valid_nodes,
    "target_cols": data_E10_4N_Thuringia.target_cols,
},  save_path)

In [105]:
# 5 Neighbours
data_E10_5N_Thuringia = make_window_data_multi_from_df(df_thuringia, tt=tt_thuringia, region_col="in_thuringia", neighbour_k=5, travel_time=10, feature_cols=["e10"], target_cols=["e10"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "01_Thuringia" / "02_E10" / "02_Time" / "data_Time_5N.pt"

torch.save({
    "x": data_E10_5N_Thuringia.x,
    "x_mask": data_E10_5N_Thuringia.x_mask,
    "y": data_E10_5N_Thuringia.y,
    "edge_index": data_E10_5N_Thuringia.edge_index,
    "edge_weight": data_E10_5N_Thuringia.edge_weight,
    "time_feat": data_E10_5N_Thuringia.time_feat,#
    "condat": data_E10_5N_Thuringia.condat,
    "train_end_idx": data_E10_5N_Thuringia.train_end_idx,
    "val_end_idx": data_E10_5N_Thuringia.val_end_idx,
    "test_end_idx": data_E10_5N_Thuringia.test_end_idx,
    "valid_nodes": data_E10_5N_Thuringia.valid_nodes,
    "target_cols": data_E10_5N_Thuringia.target_cols,
},  save_path)

In [106]:
# 6 Neighbours
data_E10_6N_Thuringia = make_window_data_multi_from_df(df_thuringia, tt=tt_thuringia, region_col="in_thuringia", neighbour_k=6, travel_time=10, feature_cols=["e10"], target_cols=["e10"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "01_Thuringia" / "02_E10" / "02_Time" / "data_Time_6N.pt"

torch.save({
    "x": data_E10_6N_Thuringia.x,
    "x_mask": data_E10_6N_Thuringia.x_mask,
    "y": data_E10_6N_Thuringia.y,
    "edge_index": data_E10_6N_Thuringia.edge_index,
    "edge_weight": data_E10_6N_Thuringia.edge_weight,
    "time_feat": data_E10_6N_Thuringia.time_feat,
    "condat": data_E10_6N_Thuringia.condat,
    "train_end_idx": data_E10_6N_Thuringia.train_end_idx,
    "val_end_idx": data_E10_6N_Thuringia.val_end_idx,
    "test_end_idx": data_E10_6N_Thuringia.test_end_idx,
    "valid_nodes": data_E10_6N_Thuringia.valid_nodes,
    "target_cols": data_E10_6N_Thuringia.target_cols,
},  save_path)

## 3.3. Data Windows Diesel

In [107]:
# 4 Neighbours
data_Diesel_4N_Thuringia = make_window_data_multi_from_df(df_thuringia, tt=tt_thuringia, region_col="in_thuringia", neighbour_k=4, travel_time=10, feature_cols=["diesel"], target_cols=["diesel"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "01_Thuringia"/ "03_Diesel" / "02_Time" / "data_Time_4N.pt"

torch.save({
    "x": data_Diesel_4N_Thuringia.x,
    "x_mask": data_Diesel_4N_Thuringia.x_mask,
    "y": data_Diesel_4N_Thuringia.y,
    "edge_index": data_Diesel_4N_Thuringia.edge_index,
    "edge_weight": data_Diesel_4N_Thuringia.edge_weight,
    "time_feat": data_Diesel_4N_Thuringia.time_feat,
    "condat": data_Diesel_4N_Thuringia.condat,
    "train_end_idx": data_Diesel_4N_Thuringia.train_end_idx,
    "val_end_idx": data_Diesel_4N_Thuringia.val_end_idx,
    "test_end_idx": data_Diesel_4N_Thuringia.test_end_idx,
    "valid_nodes": data_Diesel_4N_Thuringia.valid_nodes,
    "target_cols": data_Diesel_4N_Thuringia.target_cols,
},  save_path)

In [108]:
# 5 Neighbours
data_Diesel_5N_Thuringia = make_window_data_multi_from_df(df_thuringia, tt=tt_thuringia, region_col="in_thuringia", neighbour_k=5, travel_time=10, feature_cols=["diesel"], target_cols=["diesel"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "01_Thuringia" / "03_Diesel" / "02_Time" / "data_Time_5N.pt"

torch.save({
    "x": data_Diesel_5N_Thuringia.x,
    "x_mask": data_Diesel_5N_Thuringia.x_mask,
    "y": data_Diesel_5N_Thuringia.y,
    "edge_index": data_Diesel_5N_Thuringia.edge_index,
    "edge_weight": data_Diesel_5N_Thuringia.edge_weight,
    "time_feat": data_Diesel_5N_Thuringia.time_feat,
    "condat": data_Diesel_5N_Thuringia.condat,
    "train_end_idx": data_Diesel_5N_Thuringia.train_end_idx,
    "val_end_idx": data_Diesel_5N_Thuringia.val_end_idx,
    "test_end_idx": data_Diesel_5N_Thuringia.test_end_idx,
    "valid_nodes": data_Diesel_5N_Thuringia.valid_nodes,
    "target_cols": data_Diesel_5N_Thuringia.target_cols,
},  save_path)

In [109]:
# 6 Neighbours
data_Diesel_6N_Thuringia = make_window_data_multi_from_df(df_thuringia, tt=tt_thuringia, region_col="in_thuringia", neighbour_k=6, travel_time=10, feature_cols=["diesel"], target_cols=["diesel"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "01_Thuringia"/ "03_Diesel" / "02_Time" / "data_Time_6N.pt"

torch.save({
    "x": data_Diesel_6N_Thuringia.x,
    "x_mask": data_Diesel_6N_Thuringia.x_mask,
    "y": data_Diesel_6N_Thuringia.y,
    "edge_index": data_Diesel_6N_Thuringia.edge_index,
    "edge_weight": data_Diesel_6N_Thuringia.edge_weight,
    "time_feat": data_Diesel_6N_Thuringia.time_feat,
    "condat": data_Diesel_6N_Thuringia.condat,
    "train_end_idx": data_Diesel_6N_Thuringia.train_end_idx,
    "val_end_idx": data_Diesel_6N_Thuringia.val_end_idx,
    "test_end_idx": data_Diesel_6N_Thuringia.test_end_idx,
    "valid_nodes": data_Diesel_6N_Thuringia.valid_nodes,
    "target_cols": data_Diesel_6N_Thuringia.target_cols,
},  save_path)

# 4. Create Data Windows, Hesse

## 4.1. Data Windows E5

In [110]:
# 4 Neighbours
data_E5_4N_Hesse = make_window_data_multi_from_df(df_hesse, tt=tt_hesse, region_col="in_hesse", neighbour_k=4, travel_time=10, feature_cols=["e5"], target_cols=["e5"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "02_Hesse" / "01_E5" / "02_Time" / "data_Time_4N.pt"

torch.save({
    "x": data_E5_4N_Hesse.x,
    "x_mask": data_E5_4N_Hesse.x_mask,
    "y": data_E5_4N_Hesse.y,
    "edge_index": data_E5_4N_Hesse.edge_index,
    "edge_weight": data_E5_4N_Hesse.edge_weight,
    "time_feat": data_E5_4N_Hesse.time_feat,
    "condat": data_Diesel_6N_Hesse.condat,
    "train_end_idx": data_E5_4N_Hesse.train_end_idx,
    "val_end_idx": data_E5_4N_Hesse.val_end_idx,
    "test_end_idx": data_E5_4N_Hesse.test_end_idx,
    "valid_nodes": data_E5_4N_Hesse.valid_nodes,
    "target_cols": data_E5_4N_Hesse.target_cols,
},  save_path)

In [111]:
# 5 Neighbours
data_E5_5N_Hesse = make_window_data_multi_from_df(df_hesse, tt=tt_hesse, region_col="in_hesse", neighbour_k=5, travel_time=10, feature_cols=["e5"], target_cols=["e5"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "02_Hesse" / "01_E5" / "02_Time" / "data_Time_5N.pt"

torch.save({
    "x": data_E5_5N_Hesse.x,
    "x_mask": data_E5_5N_Hesse.x_mask,
    "y": data_E5_5N_Hesse.y,
    "edge_index": data_E5_5N_Hesse.edge_index,
    "edge_weight": data_E5_5N_Hesse.edge_weight,
    "time_feat": data_E5_5N_Hesse.time_feat,
    "condat": data_Diesel_6N_Hesse.condat,
    "train_end_idx": data_E5_5N_Hesse.train_end_idx,
    "val_end_idx": data_E5_5N_Hesse.val_end_idx,
    "test_end_idx": data_E5_5N_Hesse.test_end_idx,
    "valid_nodes": data_E5_5N_Hesse.valid_nodes,
    "target_cols": data_E5_5N_Hesse.target_cols,
},  save_path)

In [112]:
# 6 Neighbours
data_E5_6N_Hesse = make_window_data_multi_from_df(df_hesse, tt=tt_hesse, region_col="in_hesse", neighbour_k=6, travel_time=10, feature_cols=["e5"], target_cols=["e5"])
save_path = Setup.DATA_SCRATCH /"02_Processed_Data" / "03_Data_Windows" / "02_Hesse" / "01_E5" / "02_Time" / "data_Time_6N.pt"

torch.save({
    "x": data_E5_6N_Hesse.x,
    "x_mask": data_E5_6N_Hesse.x_mask,
    "y": data_E5_6N_Hesse.y,
    "edge_index": data_E5_6N_Hesse.edge_index,
    "edge_weight": data_E5_6N_Hesse.edge_weight,
    "time_feat": data_E5_6N_Hesse.time_feat,
    "condat": data_Diesel_6N_Hesse.condat,
    "train_end_idx": data_E5_6N_Hesse.train_end_idx,
    "val_end_idx": data_E5_6N_Hesse.val_end_idx,
    "test_end_idx": data_E5_6N_Hesse.test_end_idx,
    "valid_nodes": data_E5_6N_Hesse.valid_nodes,
    "target_cols": data_E5_6N_Hesse.target_cols,
},  save_path)

## 4.2. Data Windows E10

In [113]:
# 4 Neighbours
data_E10_4N_Hesse = make_window_data_multi_from_df(df_hesse, tt=tt_hesse, region_col="in_hesse", neighbour_k=4, travel_time=10, feature_cols=["e10"], target_cols=["e10"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "02_Hesse" / "02_E10" / "02_Time" / "data_Time_4N.pt"
 
torch.save({
    "x": data_E10_4N_Hesse.x,
    "x_mask": data_E10_4N_Hesse.x_mask,
    "y": data_E10_4N_Hesse.y,
    "edge_index": data_E10_4N_Hesse.edge_index,
    "edge_weight": data_E10_4N_Hesse.edge_weight,
    "time_feat": data_E10_4N_Hesse.time_feat,
    "condat": data_Diesel_6N_Hesse.condat,
    "train_end_idx": data_E10_4N_Hesse.train_end_idx,
    "val_end_idx": data_E10_4N_Hesse.val_end_idx,
    "test_end_idx": data_E10_4N_Hesse.test_end_idx,
    "valid_nodes": data_E10_4N_Hesse.valid_nodes,
    "target_cols": data_E10_4N_Hesse.target_cols,
},  save_path)

In [114]:
# 5 Neighbours
data_E10_5N_Hesse = make_window_data_multi_from_df(df_hesse, tt=tt_hesse, region_col="in_hesse", neighbour_k=5, travel_time=10, feature_cols=["e10"], target_cols=["e10"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "02_Hesse" / "02_E10" / "02_Time" / "data_Time_5N.pt"

torch.save({
    "x": data_E10_5N_Hesse.x,
    "x_mask": data_E10_5N_Hesse.x_mask,
    "y": data_E10_5N_Hesse.y,
    "edge_index": data_E10_5N_Hesse.edge_index,
    "edge_weight": data_E10_5N_Hesse.edge_weight,
    "time_feat": data_E10_5N_Hesse.time_feat,
    "condat": data_Diesel_6N_Hesse.condat,
    "train_end_idx": data_E10_5N_Hesse.train_end_idx,
    "val_end_idx": data_E10_5N_Hesse.val_end_idx,
    "test_end_idx": data_E10_5N_Hesse.test_end_idx,
    "valid_nodes": data_E10_5N_Hesse.valid_nodes,
    "target_cols": data_E10_5N_Hesse.target_cols,
},  save_path)

In [115]:
# 6 Neighbours
data_E10_6N_Hesse = make_window_data_multi_from_df(df_hesse, tt=tt_hesse, region_col="in_hesse", neighbour_k=6, travel_time=10, feature_cols=["e10"], target_cols=["e10"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "02_Hesse" / "02_E10" / "02_Time" / "data_Time_6N.pt"

torch.save({
    "x": data_E10_6N_Hesse.x,
    "x_mask": data_E10_6N_Hesse.x_mask,
    "y": data_E10_6N_Hesse.y,
    "edge_index": data_E10_6N_Hesse.edge_index,
    "edge_weight": data_E10_6N_Hesse.edge_weight,
    "time_feat": data_E10_6N_Hesse.time_feat,
    "condat": data_Diesel_6N_Hesse.condat,
    "train_end_idx": data_E10_6N_Hesse.train_end_idx,
    "val_end_idx": data_E10_6N_Hesse.val_end_idx,
    "test_end_idx": data_E10_6N_Hesse.test_end_idx,
    "valid_nodes": data_E10_6N_Hesse.valid_nodes,
    "target_cols": data_E10_6N_Hesse.target_cols,
},  save_path)

## 4.3. Data Windows Diesel

In [116]:
# 4 Neighbours
data_Diesel_4N_Hesse = make_window_data_multi_from_df(df_hesse, tt=tt_hesse, region_col="in_hesse", neighbour_k=4, travel_time=10, feature_cols=["diesel"], target_cols=["diesel"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "02_Hesse" / "03_Diesel" / "02_Time" / "data_Time_4N.pt"

torch.save({
    "x": data_Diesel_4N_Hesse.x,
    "x_mask": data_Diesel_4N_Hesse.x_mask,
    "y": data_Diesel_4N_Hesse.y,
    "edge_index": data_Diesel_4N_Hesse.edge_index,
    "edge_weight": data_Diesel_4N_Hesse.edge_weight,
    "time_feat": data_Diesel_4N_Hesse.time_feat,
    "condat": data_Diesel_6N_Hesse.condat,
    "train_end_idx": data_Diesel_4N_Hesse.train_end_idx,
    "val_end_idx": data_Diesel_4N_Hesse.val_end_idx,
    "test_end_idx": data_Diesel_4N_Hesse.test_end_idx,
    "valid_nodes": data_Diesel_4N_Hesse.valid_nodes,
    "target_cols": data_Diesel_4N_Hesse.target_cols,
},  save_path)

In [ ]:
# 5 Neighbours
data_Diesel_5N_Hesse = make_window_data_multi_from_df(df_hesse, tt=tt_hesse, region_col="in_hesse", neighbour_k=5, travel_time=10, feature_cols=["diesel"], target_cols=["diesel"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "02_Hesse" / "03_Diesel" / "02_Time" / "data_Time_5N.pt"

torch.save({
    "x": data_Diesel_5N_Hesse.x,
    "x_mask": data_Diesel_5N_Hesse.x_mask,
    "y": data_Diesel_5N_Hesse.y,
    "edge_index": data_Diesel_5N_Hesse.edge_index,
    "edge_weight": data_Diesel_5N_Hesse.edge_weight,
    "time_feat": data_Diesel_5N_Hesse.time_feat,
    "condat": data_Diesel_6N_Hesse.condat,
    "train_end_idx": data_Diesel_5N_Hesse.train_end_idx,
    "val_end_idx": data_Diesel_5N_Hesse.val_end_idx,
    "test_end_idx": data_Diesel_5N_Hesse.test_end_idx,
    "valid_nodes": data_Diesel_5N_Hesse.valid_nodes,
    "target_cols": data_Diesel_5N_Hesse.target_cols,
},  save_path)

In [ ]:
# 6 Neighbours
data_Diesel_6N_Hesse = make_window_data_multi_from_df(df_hesse, tt=tt_hesse, region_col="in_hesse", neighbour_k=6, travel_time=10, feature_cols=["diesel"], target_cols=["diesel"])
save_path = Setup.DATA_SCRATCH / "02_Processed_Data" / "03_Data_Windows" / "02_Hesse"/ "03_Diesel" / "02_Time" / "data_Time_6N.pt"

torch.save({
    "x": data_Diesel_6N_Hesse.x,
    "x_mask": data_Diesel_6N_Hesse.x_mask,
    "y": data_Diesel_6N_Hesse.y,
    "edge_index": data_Diesel_6N_Hesse.edge_index,
    "edge_weight": data_Diesel_6N_Hesse.edge_weight,
    "time_feat": data_Diesel_6N_Hesse.time_feat,
    "condat": data_Diesel_6N_Hesse.condat,
    "condat": data_Diesel_6N_Hesse.condat,
    "train_end_idx": data_Diesel_6N_Hesse.train_end_idx,
    "val_end_idx": data_Diesel_6N_Hesse.val_end_idx,
    "test_end_idx": data_Diesel_6N_Hesse.test_end_idx,
    "valid_nodes": data_Diesel_6N_Hesse.valid_nodes,
    "target_cols": data_Diesel_6N_Hesse.target_cols,
},  save_path)